In [1]:
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix,balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from collections import deque
from sklearn.decomposition import PCA
import mlflow
from sklearn.neighbors import LocalOutlierFactor


In [2]:
mlflow.set_experiment("binary_classification_mapping2_feature_engineering")

<Experiment: artifact_location='file:d:/fourth_year/graduation_project/JGuard/defenders/mult-iturn/NBF/notebooks/mlruns/5', creation_time=1780756498674, experiment_id='5', last_update_time=1780756498674, lifecycle_stage='active', name='binary_classification_mapping2_feature_engineering', tags={}, trace_location=None, workspace='default'>

In [18]:
def log_trained_model(model,y_pred,y_test,features,params: dict,run_name: str = "run",\
        model_name: str = "model",plot_name: str = "confusion_matrix.png",\
            dim_reduction_model=None):
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("features", str(features))

        # Metrics
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        f1_score_val = f1_score(y_test, y_pred, average='macro')
        classification_report_val = classification_report(y_test, y_pred, output_dict=True)
        mlflow.log_metric("balanced_accuracy", bal_acc)
        mlflow.log_metric("f1_score", f1_score_val)
        mlflow.log_dict(classification_report_val, "classification_report.json")
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.savefig(plot_name)
        plt.close()
        mlflow.log_artifact(plot_name)

        mlflow.sklearn.log_model(model, model_name)
        if dim_reduction_model is not None:
            mlflow.sklearn.log_model(dim_reduction_model, f"{model_name}_dim_reduction")

        print(f"Run complete. Balanced Accuracy: {bal_acc:.4f}")

        return {
            "balanced_accuracy": bal_acc,
            "f1_score": f1_score_val,
            "confusion_matrix": cm.tolist()  
        }

In [2]:
data=torch.load("./../train_data_with_context/all_conversations.pt")
data2=torch.load("./../train_data_with_context/all_conversations2.pt")

In [3]:
data[0][0].keys()

dict_keys(['x_t', 'zt', 'score', 'y'])

In [4]:
labels_map={
    1:0,
    2:0,
    3:0,
    4:1,
    5:1
}

In [5]:
import torch
import torch.nn.functional as F

def feature_engineering(
    x_t,
    u_t,
    x_prev,
    u_prev,
    x_prev_4step_back=None,
    u_prev_4step_back=None,
):
    features = []

    # (5/5)
    # Captures how much the internal memory/state changed from the previous turn.
    # Large values indicate abrupt context changes or possible prompt injection.
    state_drift = torch.norm(x_t - x_prev).item()
    features.append(state_drift)

    # (5/5)
    # Captures the semantic change between consecutive user prompts.
    # High values indicate topic switching or semantic drift.
    embedding_drift = torch.norm(u_t - u_prev).item()
    features.append(embedding_drift)

    # (4/5)
    # Captures disagreement between memory and current input.
    # High values may indicate context hijacking or adversarial behavior.
    state_input_distance = torch.norm(x_t - u_t).item()
    features.append(state_input_distance)

    # (4/5)
    # Captures how similar the current state is to the previous state.
    # Low similarity means the conversation memory changed significantly.
    state_similarity = F.cosine_similarity(
        x_t.unsqueeze(0),
        x_prev.unsqueeze(0)
    ).item()
    features.append(state_similarity)

    # (4/5)
    # Captures semantic continuity between consecutive prompts.
    # Low similarity means the user suddenly changed intent/topic.
    embedding_similarity = F.cosine_similarity(
        u_t.unsqueeze(0),
        u_prev.unsqueeze(0)
    ).item()
    features.append(embedding_similarity)

    # (5/5)
    # Captures alignment between current input and conversation memory.
    # Low values indicate the current prompt does not fit previous context.
    state_input_similarity = F.cosine_similarity(
        x_t.unsqueeze(0),
        u_t.unsqueeze(0)
    ).item()
    features.append(state_input_similarity)

    # (5/5)
    # Captures long-term memory drift over the last 4 turns.
    # High values indicate gradual context hijacking or accumulated deviation.
    long_term_state_drift = torch.norm(
        x_t - x_prev_4step_back
    ).item()
    features.append(long_term_state_drift)

    # (4/5)
    # Captures long-range memory consistency.
    # Low similarity indicates that the conversation has diverged
    # significantly from its earlier context.
    long_term_state_similarity = F.cosine_similarity(
        x_t.unsqueeze(0),
        x_prev_4step_back.unsqueeze(0)
    ).item()
    features.append(long_term_state_similarity)


    # (5/5)
    # Captures long-term semantic drift over 4 turns.
    # Useful for detecting slowly evolving prompt injections.
    long_term_embedding_drift = torch.norm(
        u_t - u_prev_4step_back
    ).item()
    features.append(long_term_embedding_drift)

    # (4/5)
    # Captures long-term semantic consistency.
    # Low values indicate the conversation has moved far from
    # the semantic intent several turns earlier.
    long_term_embedding_similarity = F.cosine_similarity(
        u_t.unsqueeze(0),
        u_prev_4step_back.unsqueeze(0)
    ).item()
    features.append(long_term_embedding_similarity)

    return torch.tensor(features, dtype=torch.float32)

In [6]:
X2_vecs = []
X2_features = []
y2 = []

for convo in data2:
    first_x = convo[0]["x_t"]
    first_u = convo[0]["ut"]

    x_prev = torch.zeros_like(convo[0]["x_t"])
    u_prev = torch.zeros_like(convo[0]["ut"])

    x_history = deque(maxlen=4)
    u_history = deque(maxlen=4)

    for turn in convo:
        X2_vecs.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))

        if len(x_history) == 4:
            x_prev_4step_back = x_history[0]
            u_prev_4step_back = u_history[0]
        else:
            x_prev_4step_back = first_x
            u_prev_4step_back = first_u

        handcrafted = feature_engineering(
            turn["x_t"],
            turn["ut"],
            x_prev,
            u_prev,
            x_prev_4step_back,
            u_prev_4step_back
        ).numpy()

        X2_features.append(handcrafted)
        y2.append(labels_map[turn["score"]])

        x_history.append(turn["x_t"])
        u_history.append(turn["ut"])

        x_prev = turn["x_t"]
        u_prev = turn["ut"]

In [7]:
from collections import Counter
c=Counter(y2)
c

Counter({0: 23007, 1: 5972})

In [8]:
# high imbalance ratio
ratio=c[0]/c[1]
ratio

3.852478231748158

In [9]:
# Convert to numpy arrays
X2_vecs = np.array([v.numpy() for v in X2_vecs])
X2_features = np.array(X2_features)
y2 = np.array(y2)

In [10]:
# Train-test split
X_vec_train, X_vec_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
    X2_vecs,
    X2_features,
    y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

In [11]:
lof_before = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True
)

lof_before.fit(X_vec_train)

lof_train_before = lof_before.decision_function(X_vec_train).reshape(-1, 1)
lof_test_before = lof_before.decision_function(X_vec_test).reshape(-1, 1)

# Final dataset without PCA
X_train = np.concatenate(
    [
        X_vec_train,
        X_feat_train,
        lof_train_before
    ],
    axis=1
)

X_test = np.concatenate(
    [
        X_vec_test,
        X_feat_test,
        lof_test_before
    ],
    axis=1
)

In [12]:
pca = PCA(
    n_components=0.95,
    random_state=42
)

X_vec_train_pca = pca.fit_transform(X_vec_train)
X_vec_test_pca = pca.transform(X_vec_test)

In [13]:
lof_after = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True
)

lof_after.fit(X_vec_train_pca)

lof_train_after = lof_after.decision_function(
    X_vec_train_pca
).reshape(-1, 1)

lof_test_after = lof_after.decision_function(
    X_vec_test_pca
).reshape(-1, 1)

# Final dataset with PCA
X_train_pca = np.concatenate(
    [
        X_vec_train_pca,
        X_feat_train,
        lof_train_after
    ],
    axis=1
)

X_test_pca = np.concatenate(
    [
        X_vec_test_pca,
        X_feat_test,
        lof_test_after
    ],
    axis=1
)

In [14]:
del data
del data2
del X2_vecs
del X2_features

In [47]:
svm_balanced = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")

y_train_pred = svm_balanced.fit(X_train, y_train).predict(X_train)
y_test_pred = svm_balanced.predict(X_test)

In [48]:
print("train_f1_score:", f1_score(y_train, y_train_pred, average='macro'))
print("Balanced f1_score:", f1_score(y_test, y_test_pred, average='macro'))
print(classification_report(y_test, y_test_pred))

train_f1_score: 0.7861666689605471
Balanced f1_score: 0.763864638012354
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      4602
           1       0.53      0.84      0.65      1194

    accuracy                           0.82      5796
   macro avg       0.74      0.83      0.76      5796
weighted avg       0.87      0.82      0.83      5796



In [ ]:
log_trained_model(svm_balanced,y_test_pred,y_test,features="Xt_ut",\
                  params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},\
                    run_name="SVM_rbf_balanced",model_name="svm_rbf_different",plot_name="svm_rbf_balanced_cm.png")

# with dimensionality reduction

In [50]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")  

y_train_pred=model.fit(X_train_pca, y_train).predict(X_train_pca)
y_test_pred = model.predict(X_test_pca)

In [51]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.95      0.81      0.87      4602
           1       0.53      0.84      0.65      1194

    accuracy                           0.81      5796
   macro avg       0.74      0.82      0.76      5796
weighted avg       0.86      0.81      0.83      5796



In [ ]:
log_trained_model(run_name="SVM_pca_different_weights", model=model, y_pred=y_test_pred, y_test=y_test, features="PCAXt_Zt", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_different", plot_name="svm_rbf_different_cm.png",
                dim_reduction_model=pca)

2026/06/06 15:58:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:58:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:58:41 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpnl3y39dz\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/06/06 15:58:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:58:41 WARNING mlflow.sklearn: Savi

Run complete. Balanced Accuracy: 0.8301


{'balanced_accuracy': 0.8300875302195463,
 'f1_score': 0.7729533514658258,
 'confusion_matrix': [[3782, 820], [193, 1001]]}

# Xgboost

In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report

# Compute class imbalance weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

y_train_pred = xgb_model.fit(X_train, y_train).predict(X_train)
y_test_pred = xgb_model.predict(X_test)

print("train_f1_score:", f1_score(y_train, y_train_pred, average='macro'))
print("Balanced f1_score:", f1_score(y_test, y_test_pred, average='macro'))
print(classification_report(y_test, y_test_pred))

train_f1_score: 0.9323685770928225
Balanced f1_score: 0.7768498699984769
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      4602
           1       0.60      0.72      0.65      1194

    accuracy                           0.84      5796
   macro avg       0.76      0.80      0.78      5796
weighted avg       0.86      0.84      0.85      5796



# dim reduction

In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report

# Compute class imbalance weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

y_train_pred = xgb_model.fit(X_train_pca, y_train).predict(X_train_pca)
y_test_pred = xgb_model.predict(X_test_pca)

print("train_f1_score:", f1_score(y_train, y_train_pred, average='macro'))
print("Balanced f1_score:", f1_score(y_test, y_test_pred, average='macro'))
print(classification_report(y_test, y_test_pred))

train_f1_score: 0.9289349994768326
Balanced f1_score: 0.7682434127725362
              precision    recall  f1-score   support

           0       0.92      0.87      0.89      4602
           1       0.59      0.71      0.64      1194

    accuracy                           0.84      5796
   macro avg       0.75      0.79      0.77      5796
weighted avg       0.85      0.84      0.84      5796



In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import numpy as np

# imbalance handling
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist"
)

param_dist = {
    "n_estimators": [100, 300, 500,700],
    "max_depth": [3, 4, 5, 6,8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=25,            
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Params:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

best_model = random_search.best_estimator_

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("train_f1_score:", f1_score(y_train, y_train_pred, average='macro'))
print("test_f1_score:", f1_score(y_test, y_test_pred, average='macro'))
print(classification_report(y_test, y_test_pred))

Fitting 5 folds for each of 25 candidates, totalling 125 fits


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import numpy as np

# imbalance handling
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist"
)

param_dist = {
    "n_estimators": [100, 300, 500,700],
    "max_depth": [3, 4, 5, 6,8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=25,             
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_pca, y_train)

print("Best Params:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

best_model = random_search.best_estimator_

y_train_pred = best_model.predict(X_train_pca)
y_test_pred = best_model.predict(X_test_pca)

print("train_f1_score:", f1_score(y_train, y_train_pred, average='macro'))
print("test_f1_score:", f1_score(y_test, y_test_pred, average='macro'))
print(classification_report(y_test, y_test_pred))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Params: {'subsample': 1.0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best CV Score: 0.7730779685508729
train_f1_score: 0.9820524772766275
test_f1_score: 0.7765908643250683
              precision    recall  f1-score   support

           0       0.91      0.90      0.91      4602
           1       0.64      0.65      0.65      1194

    accuracy                           0.85      5796
   macro avg       0.77      0.78      0.78      5796
weighted avg       0.85      0.85      0.85      5796

